# PlantCLEF 2015 S-CNN(A) Genus Diagnostics

Diagnostics for the already trained global-view genus model. This notebook does not train `S-CNN(B)` and does not require a reference index. It mounts Google Drive, restores the LeafScan data, finds `scnn_genus_vgg16_best.pt` on Drive, then evaluates the current `S-CNN(A)` checkpoint with several reference seeds and scoring modes.


## 1. Runtime Check

Use a GPU runtime. The diagnostics only run inference, but the all-reference oracle can still be slow on CPU.


In [1]:
import torch

print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


CUDA: True
Device: NVIDIA L4


## 2. Clone Or Update Project


In [2]:
from pathlib import Path
import os
import shutil
import subprocess

PROJECT_DIR = Path('/content/diploma')
REPO_URL = 'https://github.com/robodanill/diploma.git'
BRANCH = 'robodanill/main'


def clone_project():
    os.chdir('/content')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)


def pull_project() -> bool:
    if not (PROJECT_DIR / '.git').exists():
        return False
    result = subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
    return result.returncode == 0


if PROJECT_DIR.exists():
    print(f'Trying to update existing project: {PROJECT_DIR}')
    if not pull_project():
        print('Pull failed or project is not a git repository; cloning a fresh copy.')
        clone_project()
else:
    print(f'Project not found at {PROJECT_DIR}; cloning a fresh copy.')
    clone_project()

os.chdir(PROJECT_DIR)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.[ml]'], check=True)

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
print(f'Project commit: {commit}')


Trying to update existing project: /content/diploma
Project commit: e1665ca


## 3. Mount Google Drive


In [3]:
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Restore LeafScan Data And Paper60 Metadata

Expected archives on Drive:

- `/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz`
- `/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz`

The cell rebuilds `leafscan_paper60_metadata.csv` from official test species and adds rotation-based rows only for species with fewer than six training images, matching the training notebook setup.


In [4]:
%%bash
set -euo pipefail
trap 'echo "FAILED at line $LINENO: $BASH_COMMAND" >&2' ERR
export PYTHONUNBUFFERED=1
cd /content/diploma

ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz
TEST_ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz

echo "checking required archives"
ls -lh /content/drive/MyDrive/PlantCLEF2015*.tar.gz 2>/dev/null || true
if [ ! -f "$ARCHIVE" ]; then
  echo "Missing LeafScan training archive: $ARCHIVE" >&2
  exit 2
fi
if [ ! -f "$TEST_ARCHIVE" ]; then
  echo "Missing LeafScan test archive: $TEST_ARCHIVE" >&2
  exit 3
fi

rm -rf data/plantclef2015
mkdir -p data/plantclef2015

echo "extracting training archive: $ARCHIVE"
tar -xzf "$ARCHIVE" -C data/plantclef2015
test -f data/plantclef2015/leafscan/metadata.csv
cp data/plantclef2015/leafscan/metadata.csv data/plantclef2015/leafscan_metadata.csv

echo "extracting test archive: $TEST_ARCHIVE"
rm -rf data/plantclef2015/test_leafscan
mkdir -p data/plantclef2015/test_leafscan
tar -xzf "$TEST_ARCHIVE" -C data/plantclef2015/test_leafscan
test -f data/plantclef2015/test_leafscan/leafscan/metadata.csv
cp data/plantclef2015/test_leafscan/leafscan/metadata.csv data/plantclef2015/test_leafscan_metadata.csv

python - <<'PY2'
import csv
from collections import Counter, defaultdict
from pathlib import Path
from PIL import Image

with open('data/plantclef2015/leafscan_metadata.csv', newline='', encoding='utf-8') as file:
    source_rows = list(csv.DictReader(file))
with open('data/plantclef2015/test_leafscan_metadata.csv', newline='', encoding='utf-8') as file:
    test_rows = list(csv.DictReader(file))

test_species = {row['species'] for row in test_rows}
paper60_rows = [row for row in source_rows if row['species'] in test_species]
source_species = {row['species'] for row in source_rows}
missing_in_train = sorted(test_species - source_species)
if missing_in_train:
    raise RuntimeError(f'Missing test species in train metadata: {missing_in_train}')

fieldnames = list(source_rows[0].keys())
with open('data/plantclef2015/leafscan_paper60_metadata.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(paper60_rows)

train_count_by_species = Counter(row['species'] for row in paper60_rows)
underfilled_species = sorted(species for species in test_species if train_count_by_species[species] < 6)
if underfilled_species:
    print('paper60 species with fewer than 6 train images:', underfilled_species)
    rows_by_species = defaultdict(list)
    for row in paper60_rows:
        rows_by_species[row['species']].append(row)
    augmented_dir = Path('data/plantclef2015/leafscan/augmented')
    augmented_dir.mkdir(parents=True, exist_ok=True)
    leafscan_root = Path('data/plantclef2015/leafscan')
    angles = [180, 90, 270, 15, -15]
    augmented_rows = []
    for species in underfilled_species:
        species_rows = rows_by_species[species]
        if not species_rows:
            raise RuntimeError(f'Cannot augment {species}: no train rows found')
        needed = 6 - len(species_rows)
        for index in range(needed):
            base_row = species_rows[index % len(species_rows)]
            source_path = Path(base_row['image_path'])
            if not source_path.is_absolute():
                source_path = leafscan_root / source_path
            angle = angles[index % len(angles)]
            output_name = f"{source_path.stem}_aug_rot{angle}_{index + 1}.jpg".replace('-', 'm')
            output_path = augmented_dir / output_name
            with Image.open(source_path) as image:
                image.convert('RGB').rotate(angle, expand=True, fillcolor=(255, 255, 255)).save(output_path, quality=95)
            augmented_row = dict(base_row)
            augmented_row['image_path'] = str(output_path.relative_to(leafscan_root))
            if 'source_xml' in augmented_row:
                augmented_row['source_xml'] = f"{augmented_row['source_xml']}#aug_rot{angle}"
            augmented_rows.append(augmented_row)
    paper60_rows.extend(augmented_rows)
    with open('data/plantclef2015/leafscan_paper60_metadata.csv', 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(paper60_rows)
    print('paper60 augmented rows added:', len(augmented_rows))

train_count_by_species = Counter(row['species'] for row in paper60_rows)
underfilled_species = sorted(species for species in test_species if train_count_by_species[species] < 6)
if underfilled_species:
    raise RuntimeError(f'Cannot build 6-shot paper subset: {underfilled_species}')

print('leafscan source rows:', len(source_rows))
print('paper60 train rows:', len(paper60_rows))
print('paper60 train genera:', len({row['genus'] for row in paper60_rows}))
print('paper60 train species:', len({row['species'] for row in paper60_rows}))
paper60_species_by_genus = defaultdict(set)
test_species_by_genus = defaultdict(set)
for row in paper60_rows:
    paper60_species_by_genus[row['genus']].add(row['species'])
for row in test_rows:
    test_species_by_genus[row['genus']].add(row['species'])
paper60_species_per_genus = {genus: len(species) for genus, species in paper60_species_by_genus.items()}
test_species_per_genus = {genus: len(species) for genus, species in test_species_by_genus.items()}
print('paper60 max species per genus:', max(paper60_species_per_genus.values()))
print('paper60 genera with >6 species:', sorted(g for g, count in paper60_species_per_genus.items() if count > 6))
print('paper60 test rows:', len(test_rows))
print('paper60 test genera:', len({row['genus'] for row in test_rows}))
print('paper60 test species:', len(test_species))
print('paper60 test max species per genus:', max(test_species_per_genus.values()))
print('paper60 test genera with >6 species:', sorted(g for g, count in test_species_per_genus.items() if count > 6))
print('paper60 six-shot training rows:', 6 * len(test_species))
print('smallest train species counts:', train_count_by_species.most_common()[-10:])
PY2


checking required archives
-rw------- 1 root root 1.2G May  3 18:06 /content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz
-rw------- 1 root root  22M May  3 16:51 /content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz
-rw------- 1 root root    0 May  3 16:41 /content/drive/MyDrive/PlantCLEF2015TestDataWithAnnotations.tar.gz
extracting training archive: /content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz
extracting test archive: /content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz
paper60 species with fewer than 6 train images: ['Artemisia vulgaris L.']
paper60 augmented rows added: 1
leafscan source rows: 12605
paper60 train rows: 6555
paper60 train genera: 43
paper60 train species: 60
paper60 max species per genus: 6
paper60 genera with >6 species: []
paper60 test rows: 221
paper60 test genera: 43
paper60 test species: 60
paper60 test max species per genus: 6
paper60 test genera with >6 species: []
paper60 six-shot training rows: 360
smallest train species counts: [(

## 5. Locate The Genus Checkpoint On Drive

By default this selects the newest `scnn_genus_vgg16_best.pt` found under `/content/drive/MyDrive/diploma_checkpoints`. Set `MANUAL_GENUS_CHECKPOINT` if you want a specific file.


In [5]:
from pathlib import Path
import shutil as shutil_module

MANUAL_GENUS_CHECKPOINT = ''
DRIVE_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/diploma_checkpoints')
LOCAL_CHECKPOINT = Path('/content/diploma/checkpoints/scnn_genus_vgg16_best.pt')

if MANUAL_GENUS_CHECKPOINT:
    checkpoint = Path(MANUAL_GENUS_CHECKPOINT)
else:
    patterns = [
        'leafscan_vgg16/**/scnn_genus_vgg16_best.pt',
        '**/scnn_genus_vgg16_best.pt',
        'leafscan_vgg16/**/scnn_genus_vgg16.pt',
        '**/scnn_genus_vgg16.pt',
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(DRIVE_CHECKPOINT_ROOT.glob(pattern))
    candidates = sorted(set(candidates), key=lambda path: path.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f'No VGG16 genus checkpoint found under {DRIVE_CHECKPOINT_ROOT}')
    checkpoint = candidates[0]

if not checkpoint.exists():
    raise FileNotFoundError(checkpoint)

LOCAL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
shutil_module.copy2(checkpoint, LOCAL_CHECKPOINT)
Path('/content/diploma/.genus_checkpoint_path').write_text(str(LOCAL_CHECKPOINT), encoding='utf-8')
print('Selected Drive checkpoint:', checkpoint)
print('Copied to:', LOCAL_CHECKPOINT)
print('Size:', LOCAL_CHECKPOINT.stat().st_size)


Selected Drive checkpoint: /content/drive/MyDrive/diploma_checkpoints/leafscan_vgg16/genus_20260524T123445Z_24567/scnn_genus_vgg16_best.pt
Copied to: /content/diploma/checkpoints/scnn_genus_vgg16_best.pt
Size: 537071831


## 6. Evaluate Full VGG16 S-CNN(A)


In [8]:
from pathlib import Path
import os
import subprocess

PROJECT_DIR = Path('/content/diploma')
checkpoint = Path((PROJECT_DIR / '.genus_checkpoint_path').read_text(encoding='utf-8').strip())
REFERENCE_SEED = 42
TOP_K = [5, 15, 30, 50]

cmd = [
    'python', '-u', '-m', 'plant_classifier.training.eval_genus_cli',
    '--config', 'configs/leafscan_paper60_training.yaml',
    '--query-config', 'configs/leafscan_test.yaml',
    '--checkpoint', str(checkpoint),
    '--max-species', '0',
    '--references-per-genus', '6',
    '--reference-level', 'genus',
    '--reference-seed', str(REFERENCE_SEED),
    '--reference-split', 'train',
    '--score-mode', 'comparator',
    '--top-k', *[str(value) for value in TOP_K],
]

print('checkpoint:', checkpoint)
print('reference_seed:', REFERENCE_SEED)
print('score_mode: comparator')
os.environ['PYTHONUNBUFFERED'] = '1'
print(' '.join(cmd))

!python -u -m plant_classifier.training.eval_genus_cli --config configs/leafscan_paper60_training.yaml --query-config configs/leafscan_test.yaml --checkpoint /content/diploma/checkpoints/scnn_genus_vgg16_best.pt --max-species 0 --references-per-genus 6 --reference-level genus --reference-seed 42 --reference-split train --score-mode comparator --top-k 5 15 30 50
# subprocess.run(cmd, cwd=PROJECT_DIR, check=True)


checkpoint: /content/diploma/checkpoints/scnn_genus_vgg16_best.pt
reference_seed: 42
score_mode: comparator
python -u -m plant_classifier.training.eval_genus_cli --config configs/leafscan_paper60_training.yaml --query-config configs/leafscan_test.yaml --checkpoint /content/diploma/checkpoints/scnn_genus_vgg16_best.pt --max-species 0 --references-per-genus 6 --reference-level genus --reference-seed 42 --reference-split train --score-mode comparator --top-k 5 15 30 50
using reference subset: 360 images from 60 species
references=258 queries=221 top_k=[5, 15, 30, 50] score_mode=comparator reference_seed=42
top5_genus_accuracy=0.484 (107/221)
top15_genus_accuracy=0.606 (134/221)
top30_genus_accuracy=0.706 (156/221)
top50_genus_accuracy=0.769 (170/221)
query genus distribution: {'Acer': 27, 'Ruscus': 25, 'Fraxinus': 17, 'Viburnum': 12, 'Quercus': 11, 'Ginkgo': 9, 'Hedera': 9, 'Castanea': 9, 'Populus': 8, 'Lippia': 7}
reference genus distribution: {'Acer': 6, 'Ailanthus': 6, 'Alnus': 6, 'Art

## 7. Diagnostic VGG16 S-CNN(A) L1 Ranking


In [9]:
from pathlib import Path
import os
import subprocess

PROJECT_DIR = Path('/content/diploma')
checkpoint = Path((PROJECT_DIR / '.genus_checkpoint_path').read_text(encoding='utf-8').strip())
REFERENCE_SEED = 42
TOP_K = [5, 15, 30, 50]

cmd = [
    'python', '-u', '-m', 'plant_classifier.training.eval_genus_cli',
    '--config', 'configs/leafscan_paper60_training.yaml',
    '--query-config', 'configs/leafscan_test.yaml',
    '--checkpoint', str(checkpoint),
    '--max-species', '0',
    '--references-per-genus', '6',
    '--reference-level', 'genus',
    '--reference-seed', str(REFERENCE_SEED),
    '--reference-split', 'train',
    '--score-mode', 'l1',
    '--top-k', *[str(value) for value in TOP_K],
]

print('checkpoint:', checkpoint)
print('reference_seed:', REFERENCE_SEED)
print('score_mode: l1')
os.environ['PYTHONUNBUFFERED'] = '1'
print(' '.join(cmd))
!python -u -m plant_classifier.training.eval_genus_cli --config configs/leafscan_paper60_training.yaml --query-config configs/leafscan_test.yaml --checkpoint /content/diploma/checkpoints/scnn_genus_vgg16_best.pt --max-species 0 --references-per-genus 6 --reference-level genus --reference-seed 42 --reference-split train --score-mode l1 --top-k 5 15 30 50
# eval(' '.join(cmd))
# subprocess.run(cmd, cwd=PROJECT_DIR, check=True)


checkpoint: /content/diploma/checkpoints/scnn_genus_vgg16_best.pt
reference_seed: 42
score_mode: l1
python -u -m plant_classifier.training.eval_genus_cli --config configs/leafscan_paper60_training.yaml --query-config configs/leafscan_test.yaml --checkpoint /content/diploma/checkpoints/scnn_genus_vgg16_best.pt --max-species 0 --references-per-genus 6 --reference-level genus --reference-seed 42 --reference-split train --score-mode l1 --top-k 5 15 30 50
using reference subset: 360 images from 60 species
references=258 queries=221 top_k=[5, 15, 30, 50] score_mode=l1 reference_seed=42
top5_genus_accuracy=0.552 (122/221)
top15_genus_accuracy=0.747 (165/221)
top30_genus_accuracy=0.810 (179/221)
top50_genus_accuracy=0.864 (191/221)
query genus distribution: {'Acer': 27, 'Ruscus': 25, 'Fraxinus': 17, 'Viburnum': 12, 'Quercus': 11, 'Ginkgo': 9, 'Hedera': 9, 'Castanea': 9, 'Populus': 8, 'Lippia': 7}
reference genus distribution: {'Acer': 6, 'Ailanthus': 6, 'Alnus': 6, 'Artemisia': 6, 'Asplenium':

## 8. Epoch Checkpoint Sweep For S-CNN(A)

Evaluates saved genus checkpoints `scnn_genus_vgg16_10.pt`, `scnn_genus_vgg16_20.pt`, ... on the official LeafScan test split and compares them with the same reference protocol.


In [7]:
from pathlib import Path
import os
import re
import subprocess
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

PROJECT_DIR = Path('/content/diploma')
DRIVE_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/diploma_checkpoints')
LOCAL_CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints'
MANUAL_GENUS_CHECKPOINT_DIR = ''
REFERENCE_SEED = 42
TOP_K = [5, 15, 30, 50]
SCORE_MODES = ['comparator', 'l1']
INCLUDE_BEST_AND_FINAL = True


def checkpoint_epoch(path: Path):
    match = re.search(r'_([0-9]+)\.pt$', path.name)
    return int(match.group(1)) if match else None


def newest_epoch_checkpoint_dir() -> Path | None:
    if MANUAL_GENUS_CHECKPOINT_DIR:
        manual = Path(MANUAL_GENUS_CHECKPOINT_DIR)
        if not manual.exists():
            raise FileNotFoundError(manual)
        return manual

    candidates = []
    for root in [DRIVE_CHECKPOINT_ROOT / 'leafscan_vgg16', DRIVE_CHECKPOINT_ROOT]:
        if not root.exists():
            continue
        for path in root.glob('**/scnn_genus_vgg16_[0-9]*.pt'):
            candidates.append(path.parent)
    candidates = sorted(set(candidates), key=lambda path: path.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None


def collect_checkpoints() -> list[Path]:
    checkpoint_dir = newest_epoch_checkpoint_dir()
    paths = []
    if checkpoint_dir is not None:
        print('epoch checkpoint dir:', checkpoint_dir)
        paths.extend(checkpoint_dir.glob('scnn_genus_vgg16_[0-9]*.pt'))
        if INCLUDE_BEST_AND_FINAL:
            paths.extend(
                path for path in [
                    checkpoint_dir / 'scnn_genus_vgg16_best.pt',
                    checkpoint_dir / 'scnn_genus_vgg16.pt',
                ]
                if path.exists()
            )
    else:
        print('No epoch checkpoints found on Drive; checking local checkpoints directory.')
        paths.extend(LOCAL_CHECKPOINT_DIR.glob('scnn_genus_vgg16_[0-9]*.pt'))
        if INCLUDE_BEST_AND_FINAL:
            paths.extend(
                path for path in [
                    LOCAL_CHECKPOINT_DIR / 'scnn_genus_vgg16_best.pt',
                    LOCAL_CHECKPOINT_DIR / 'scnn_genus_vgg16.pt',
                ]
                if path.exists()
            )

    unique_paths = sorted(
        set(paths),
        key=lambda path: (
            checkpoint_epoch(path) is None,
            checkpoint_epoch(path) if checkpoint_epoch(path) is not None else 10**9,
            path.name,
            str(path),
        ),
    )
    numbered = [path for path in unique_paths if checkpoint_epoch(path) is not None]
    print(f'numbered checkpoints: {len(numbered)}')
    print(f'total checkpoints to evaluate: {len(unique_paths)}')
    for path in unique_paths:
        print(' ', path.name, path)
    if not numbered:
        raise FileNotFoundError('No scnn_genus_vgg16_[0-9]*.pt checkpoints found. Run training with checkpoint_every_epochs first and save checkpoints to Drive.')
    return unique_paths


def parse_eval_output(text: str) -> dict:
    result = {}
    header = re.search(r'references=(\d+) queries=(\d+)', text)
    if header:
        result['references'] = int(header.group(1))
        result['queries'] = int(header.group(2))
    for match in re.finditer(r'top(\d+)_genus_accuracy=([0-9.]+) \((\d+)/(\d+)\)', text):
        top_k = match.group(1)
        result[f'top{top_k}'] = float(match.group(2))
        result[f'top{top_k}_hits'] = int(match.group(3))
        result[f'top{top_k}_total'] = int(match.group(4))
    return result


rows = []
checkpoints = collect_checkpoints()
for checkpoint in checkpoints:
    epoch = checkpoint_epoch(checkpoint)
    if checkpoint.name.endswith('_best.pt'):
        checkpoint_kind = 'best'
    elif checkpoint.name == 'scnn_genus_vgg16.pt':
        checkpoint_kind = 'final'
    else:
        checkpoint_kind = f'epoch_{epoch}'

    for score_mode in SCORE_MODES:
        cmd = [
            'python', '-u', '-m', 'plant_classifier.training.eval_genus_cli',
            '--config', 'configs/leafscan_paper60_training.yaml',
            '--query-config', 'configs/leafscan_test.yaml',
            '--checkpoint', str(checkpoint),
            '--max-species', '0',
            '--references-per-genus', '6',
            '--reference-level', 'genus',
            '--reference-seed', str(REFERENCE_SEED),
            '--reference-split', 'train',
            '--score-mode', score_mode,
            '--top-k', *[str(value) for value in TOP_K],
        ]
        print('\n===', checkpoint_kind, checkpoint.name, score_mode, '===')
        completed = subprocess.run(
            cmd,
            cwd=PROJECT_DIR,
            text=True,
            capture_output=True,
            env={**os.environ, 'PYTHONUNBUFFERED': '1'},
        )
        print(completed.stdout)
        if completed.returncode != 0:
            print(completed.stderr)
            completed.check_returncode()
        parsed = parse_eval_output(completed.stdout)
        rows.append(
            {
                'checkpoint_kind': checkpoint_kind,
                'epoch': epoch,
                'score_mode': score_mode,
                'checkpoint_name': checkpoint.name,
                'checkpoint_path': str(checkpoint),
                **parsed,
            }
        )

summary = pd.DataFrame(rows)
sort_cols = ['score_mode', 'top15', 'top5', 'top30']
summary = summary.sort_values(sort_cols, ascending=[True, False, False, False]).reset_index(drop=True)

out_dir = Path('/content/drive/MyDrive/diploma_diagnostics/genus_a')
if not out_dir.parent.exists():
    out_dir = PROJECT_DIR / 'diagnostics/genus_a'
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_path = out_dir / f'genus_epoch_checkpoint_sweep_{stamp}.csv'
latest_path = out_dir / 'genus_epoch_checkpoint_sweep_latest.csv'
summary.to_csv(out_path, index=False)
summary.to_csv(latest_path, index=False)

print('Saved:', out_path)
print('Saved:', latest_path)
print('Best rows by score mode:')
display(
    summary.sort_values(['score_mode', 'top15', 'top5'], ascending=[True, False, False])
    .groupby('score_mode')
    .head(5)
)
print('All rows:')
display(summary)


No epoch checkpoints found on Drive; checking local checkpoints directory.
numbered checkpoints: 9
total checkpoints to evaluate: 11
  scnn_genus_vgg16_10.pt /content/diploma/checkpoints/scnn_genus_vgg16_10.pt
  scnn_genus_vgg16_20.pt /content/diploma/checkpoints/scnn_genus_vgg16_20.pt
  scnn_genus_vgg16_30.pt /content/diploma/checkpoints/scnn_genus_vgg16_30.pt
  scnn_genus_vgg16_40.pt /content/diploma/checkpoints/scnn_genus_vgg16_40.pt
  scnn_genus_vgg16_50.pt /content/diploma/checkpoints/scnn_genus_vgg16_50.pt
  scnn_genus_vgg16_60.pt /content/diploma/checkpoints/scnn_genus_vgg16_60.pt
  scnn_genus_vgg16_70.pt /content/diploma/checkpoints/scnn_genus_vgg16_70.pt
  scnn_genus_vgg16_80.pt /content/diploma/checkpoints/scnn_genus_vgg16_80.pt
  scnn_genus_vgg16_90.pt /content/diploma/checkpoints/scnn_genus_vgg16_90.pt
  scnn_genus_vgg16.pt /content/diploma/checkpoints/scnn_genus_vgg16.pt
  scnn_genus_vgg16_best.pt /content/diploma/checkpoints/scnn_genus_vgg16_best.pt

=== epoch_10 scnn_gen

,checkpoint_kind,epoch,score_mode,checkpoint_name,checkpoint_path,references,queries,top5,top5_hits,top5_total,top15,top15_hits,top15_total,top30,top30_hits,top30_total,top50,top50_hits,top50_total
0,epoch_30,30.0,comparator,scnn_genus_vgg16_30.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.498,110,221,0.683,151,221,0.742,164,221,0.824,182,221
1,best,NaN,comparator,scnn_genus_vgg16_best.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.457,101,221,0.656,145,221,0.733,162,221,0.805,178,221
2,epoch_20,20.0,comparator,scnn_genus_vgg16_20.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.443,98,221,0.643,142,221,0.765,169,221,0.837,185,221
3,epoch_50,50.0,comparator,scnn_genus_vgg16_50.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.443,98,221,0.629,139,221,0.733,162,221,0.801,177,221
4,epoch_40,40.0,comparator,scnn_genus_vgg16_40.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.448,99,221,0.620,137,221,0.742,164,221,0.805,178,221
11,epoch_10,10.0,l1,scnn_genus_vgg16_10.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.593,131,221,0.765,169,221,0.873,193,221,0.937,207,221
12,epoch_30,30.0,l1,scnn_genus_vgg16_30.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.629,139,221,0.760,168,221,0.837,185,221,0.873,193,221
13,epoch_20,20.0,l1,scnn_genus_vgg16_20.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.606,134,221,0.756,167,221,0.842,186,221,0.878,194,221
14,epoch_50,50.0,l1,scnn_genus_vgg16_50.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.602,133,221,0.751,166,221,0.819,181,221,0.842,186,221
15,best,NaN,l1,scnn_genus_vgg16_best.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.597,132,221,0.751,166,221,0.814,180,221,0.855,189,221


All rows:


,checkpoint_kind,epoch,score_mode,checkpoint_name,checkpoint_path,references,queries,top5,top5_hits,top5_total,top15,top15_hits,top15_total,top30,top30_hits,top30_total,top50,top50_hits,top50_total
0,epoch_30,30.0,comparator,scnn_genus_vgg16_30.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.498,110,221,0.683,151,221,0.742,164,221,0.824,182,221
1,best,NaN,comparator,scnn_genus_vgg16_best.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.457,101,221,0.656,145,221,0.733,162,221,0.805,178,221
2,epoch_20,20.0,comparator,scnn_genus_vgg16_20.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.443,98,221,0.643,142,221,0.765,169,221,0.837,185,221
3,epoch_50,50.0,comparator,scnn_genus_vgg16_50.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.443,98,221,0.629,139,221,0.733,162,221,0.801,177,221
4,epoch_40,40.0,comparator,scnn_genus_vgg16_40.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.448,99,221,0.620,137,221,0.742,164,221,0.805,178,221
5,epoch_60,60.0,comparator,scnn_genus_vgg16_60.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.466,103,221,0.593,131,221,0.715,158,221,0.783,173,221
6,epoch_80,80.0,comparator,scnn_genus_vgg16_80.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.471,104,221,0.579,128,221,0.724,160,221,0.792,175,221
7,epoch_90,90.0,comparator,scnn_genus_vgg16_90.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.462,102,221,0.579,128,221,0.733,162,221,0.787,174,221
8,final,NaN,comparator,scnn_genus_vgg16.pt,/content/diploma/checkpoints/scnn_genus_vgg16.pt,258,221,0.457,101,221,0.575,127,221,0.729,161,221,0.796,176,221
9,epoch_70,70.0,comparator,scnn_genus_vgg16_70.pt,/content/diploma/checkpoints/scnn_genus_vgg16_...,258,221,0.471,104,221,0.566,125,221,0.738,163,221,0.792,175,221


## 9. Seed Sweep For Paper-Style Genus Retrieval

This is the main diagnostic for the current `S-CNN(A)` checkpoint. `Rk` is interpreted as top-k reference entries, matching the paper. Each run selects six genus references with a different seed and evaluates both the trained comparator head and raw L1 embedding distance.


In [ ]:
from pathlib import Path
import re
import subprocess
import pandas as pd

PROJECT_DIR = Path('/content/diploma')
checkpoint = Path((PROJECT_DIR / '.genus_checkpoint_path').read_text(encoding='utf-8').strip())
seeds = [1, 2, 3, 4, 5, 42, 123, 2024]
score_modes = ['comparator', 'l1']
top_ks = [5, 15, 30, 50]
rows = []

for score_mode in score_modes:
    for seed in seeds:
        cmd = [
            'python', '-u', '-m', 'plant_classifier.training.eval_genus_cli',
            '--config', 'configs/leafscan_paper60_training.yaml',
            '--query-config', 'configs/leafscan_test.yaml',
            '--checkpoint', str(checkpoint),
            '--max-species', '0',
            '--references-per-genus', '6',
            '--reference-level', 'genus',
            '--reference-seed', str(seed),
            '--reference-split', 'train',
            '--score-mode', score_mode,
            '--top-k', *[str(value) for value in top_ks],
        ]
        print('\n===', score_mode, 'seed', seed, '===')
        output = subprocess.check_output(cmd, cwd=PROJECT_DIR, text=True)
        print(output)
        row = {'score_mode': score_mode, 'reference_seed': seed}
        for top_k, accuracy in re.findall(r'top(\d+)_genus_accuracy=([0-9.]+)', output):
            row[f'top{top_k}'] = float(accuracy)
        refs = re.search(r'references=(\d+) queries=(\d+)', output)
        if refs:
            row['references'] = int(refs.group(1))
            row['queries'] = int(refs.group(2))
        rows.append(row)

results = pd.DataFrame(rows)
display(results)
summary = results.groupby('score_mode')[[f'top{k}' for k in top_ks]].agg(['mean', 'std', 'min', 'max']).round(3)
display(summary)

out_dir = Path('/content/drive/MyDrive/diploma_diagnostics/genus_a')
out_dir.mkdir(parents=True, exist_ok=True)
results.to_csv(out_dir / 'genus_seed_sweep_latest.csv', index=False)
print('Saved:', out_dir / 'genus_seed_sweep_latest.csv')


## 10. All-Reference Oracle Diagnostic

This uses all available paper60 train rows as genus references instead of six references per genus. It is not the paper protocol. It answers a different question: whether the learned embedding/checkpoint can find the correct genus when reference selection is no longer the bottleneck.


In [10]:
from pathlib import Path
import re
import subprocess
import pandas as pd

PROJECT_DIR = Path('/content/diploma')
checkpoint = Path((PROJECT_DIR / '.genus_checkpoint_path').read_text(encoding='utf-8').strip())
rows = []

for score_mode in ['comparator', 'l1']:
    cmd = [
        'python', '-u', '-m', 'plant_classifier.training.eval_genus_cli',
        '--config', 'configs/leafscan_paper60_training.yaml',
        '--query-config', 'configs/leafscan_test.yaml',
        '--checkpoint', str(checkpoint),
        '--max-species', '0',
        '--references-per-genus', '9999',
        '--reference-level', 'genus',
        '--reference-split', 'train',
        '--use-full-reference-pool',
        '--score-mode', score_mode,
        '--top-k', '5', '15', '30', '50', '100', '200',
    ]
    print('\n=== all-reference oracle', score_mode, '===')
    output = subprocess.check_output(cmd, cwd=PROJECT_DIR, text=True)
    print(output)
    row = {'score_mode': score_mode}
    for top_k, accuracy in re.findall(r'top(\d+)_genus_accuracy=([0-9.]+)', output):
        row[f'top{top_k}'] = float(accuracy)
    refs = re.search(r'references=(\d+) queries=(\d+)', output)
    if refs:
        row['references'] = int(refs.group(1))
        row['queries'] = int(refs.group(2))
    rows.append(row)

oracle = pd.DataFrame(rows)
display(oracle)
out_dir = Path('/content/drive/MyDrive/diploma_diagnostics/genus_a')
out_dir.mkdir(parents=True, exist_ok=True)
oracle.to_csv(out_dir / 'genus_all_reference_oracle_latest.csv', index=False)
print('Saved:', out_dir / 'genus_all_reference_oracle_latest.csv')



=== all-reference oracle comparator ===
references=6555 queries=221 top_k=[5, 15, 30, 50, 100, 200] score_mode=comparator reference_seed=None
top5_genus_accuracy=0.498 (110/221)
top15_genus_accuracy=0.679 (150/221)
top30_genus_accuracy=0.765 (169/221)
top50_genus_accuracy=0.828 (183/221)
top100_genus_accuracy=0.869 (192/221)
top200_genus_accuracy=0.932 (206/221)
query genus distribution: {'Acer': 27, 'Ruscus': 25, 'Fraxinus': 17, 'Viburnum': 12, 'Quercus': 11, 'Ginkgo': 9, 'Hedera': 9, 'Castanea': 9, 'Populus': 8, 'Lippia': 7}
reference genus distribution: {'Acer': 591, 'Populus': 516, 'Ulmus': 382, 'Olea': 311, 'Quercus': 310, 'Viburnum': 303, 'Ruscus': 288, 'Fraxinus': 265, 'Hedera': 264, 'Crataegus': 247}
reference pool species per genus: {'Acer': 6, 'Quercus': 4, 'Fraxinus': 3, 'Populus': 3, 'Prunus': 3, 'Crataegus': 2, 'Tilia': 2, 'Viburnum': 2, 'Ailanthus': 1, 'Alnus': 1}
genus reference species coverage: {'Acer': '6/6', 'Quercus': '4/4', 'Fraxinus': '3/3', 'Populus': '3/3', 'Pr

,score_mode,top5,top15,top30,top50,top100,top200,references,queries
0,comparator,0.498,0.679,0.765,0.828,0.869,0.932,6555,221
1,l1,0.652,0.783,0.810,0.842,0.896,0.937,6555,221


Saved: /content/drive/MyDrive/diploma_diagnostics/genus_a/genus_all_reference_oracle_latest.csv


## 11. Error Analysis For One Run

This lists the genera that fail to enter `Rk=30` most often for a selected scoring mode and reference seed.


In [12]:
from collections import Counter
from pathlib import Path
import os
import sys

import torch
import yaml

PROJECT_DIR = Path('/content/diploma')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))

from plant_classifier.data import filter_records_by_split, limit_records_by_species, load_metadata_csv  # noqa: E402
from plant_classifier.models.siamese import BackboneSpec, build_siamese_network  # noqa: E402
from plant_classifier.training.genus_eval import (  # noqa: E402
    describe_genus_reference_coverage,
    describe_species_per_genus,
    embed_image,
    rank_references,
    select_reference_records,
)
from plant_classifier.training.image_pairs import build_image_transform  # noqa: E402

checkpoint = Path((PROJECT_DIR / '.genus_checkpoint_path').read_text(encoding='utf-8').strip())
SCORE_MODE = 'l1'
REFERENCE_SEED = 42
RK = 30


def load_config(path):
    with open(path, 'r', encoding='utf-8') as file:
        return yaml.safe_load(file)


def load_records(dataset_config):
    return load_metadata_csv(
        metadata_path=Path(dataset_config['metadata']),
        dataset_root=Path(dataset_config['root']),
        image_column=dataset_config['image_column'],
        family_column=dataset_config['family_column'],
        genus_column=dataset_config['genus_column'],
        species_column=dataset_config['species_column'],
    )


def preprocessing_enabled(config):
    preprocessing = config.get('preprocessing', {})
    return bool(preprocessing.get('enabled', preprocessing.get('leaf_bbox', False)))


def apply_subset(records, dataset_config):
    subset = dataset_config.get('subset')
    if not subset:
        return records
    return limit_records_by_species(
        records,
        max_species=subset.get('max_species'),
        min_images_per_species=int(subset.get('min_images_per_species', 1)),
        max_images_per_species=subset.get('max_images_per_species'),
        seed=subset.get('seed'),
    )

config = load_config(PROJECT_DIR / 'configs/leafscan_paper60_training.yaml')
query_config = load_config(PROJECT_DIR / 'configs/leafscan_test.yaml')
records = apply_subset(load_records(config['dataset']), config['dataset'])
query_records = load_records(query_config['dataset'])
query_species = {record.species for record in query_records}
reference_records = [
    record for record in filter_records_by_split(records, 'train')
    if record.species in query_species
]
references = select_reference_records(
    reference_records,
    taxonomic_level='genus',
    references_per_label=6,
    seed=REFERENCE_SEED,
)
print('reference pool species per genus:', describe_species_per_genus(reference_records))
print('genus reference species coverage:', describe_genus_reference_coverage(reference_records, references))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_siamese_network(BackboneSpec(name=config['model']['backbone'], pretrained=False)).to(device)
model.load_state_dict(torch.load(checkpoint, map_location=device))
model.eval()
transform = build_image_transform(
    'global',
    image_size=int(config['views']['global']['image_size']),
    crop_size=int(config['views']['local']['crop_size']),
    preprocessing=preprocessing_enabled(config),
)
reference_embeddings = [(record, embed_image(model, record.image_path, transform, device)) for record in references]

misses = []
confusions = Counter()
for query in query_records:
    query_embedding = embed_image(model, query.image_path, transform, device)
    ranked = rank_references(model, query_embedding, reference_embeddings, score_mode=SCORE_MODE)
    top_genera = [record.genus for record, _score in ranked[:RK]]
    if query.genus not in top_genera:
        misses.append(query)
        confusions.update(top_genera[:5])

print(f'score_mode={SCORE_MODE} reference_seed={REFERENCE_SEED} Rk={RK}')
print(f'misses={len(misses)}/{len(query_records)} accuracy={(len(query_records) - len(misses)) / len(query_records):.3f}')
print('missed query genera:', Counter(record.genus for record in misses).most_common(30))
print('top wrong genera among first 5 positions for misses:', confusions.most_common(30))
print('first missed samples:')
for record in misses[:30]:
    print(record.genus, '|', record.species, '|', record.image_path)


reference pool species per genus: {'Acer': 6, 'Quercus': 4, 'Fraxinus': 3, 'Populus': 3, 'Prunus': 3, 'Crataegus': 2, 'Tilia': 2, 'Viburnum': 2, 'Ailanthus': 1, 'Alnus': 1}
genus reference species coverage: {'Acer': '6/6', 'Quercus': '4/4', 'Fraxinus': '3/3', 'Populus': '3/3', 'Prunus': '3/3', 'Crataegus': '2/2', 'Tilia': '2/2', 'Viburnum': '2/2', 'Ailanthus': '1/1', 'Alnus': '1/1'}
score_mode=l1 reference_seed=42 Rk=30
misses=42/221 accuracy=0.810
missed query genera: [('Fraxinus', 9), ('Ailanthus', 4), ('Carpinus', 4), ('Tilia', 3), ('Populus', 2), ('Sorbus', 2), ('Betula', 2), ('Prunus', 2), ('Crataegus', 1), ('Broussonetia', 1), ('Acer', 1), ('Syringa', 1), ('Platanus', 1), ('Geranium', 1), ('Robinia', 1), ('Olea', 1), ('Ilex', 1), ('Salix', 1), ('Viburnum', 1), ('Atriplex', 1), ('Ruscus', 1), ('Quercus', 1)]
top wrong genera among first 5 positions for misses: [('Acer', 30), ('Populus', 28), ('Liquidambar', 23), ('Sorbus', 15), ('Viburnum', 14), ('Ficus', 12), ('Buddleja', 7), ('L